## Topic: End-to-End ML Project (Real Workflow)
## Goal of Today

By the end of this session, we will:

- Build a full ML project (real workflow)
- Apply EDA → Cleaning → Feature Engineering → Model → Tuning → - - - - Evaluation
- Save and reuse the final system
- Prepare a project for GitHub

### 1. Project Overview
Project:

👉 Customer Purchase Prediction (Advanced Version)

Features:
- Age
- Salary
- Experience
- Gender

Target:
- Purchased (0 / 1)

### 2. Full Pipeline
```
Raw Data → Cleaning → Encoding → Feature Engineering → Pipeline → GridSearch → Evaluation → Save Model
```
Tools You Will Use
- Pandas
- Scikit-learn
- joblib

This is your first complete professional ML pipeline project.
I’ll now explain every step in a very simple + conceptual + interview-ready way.

### Project Goal (Big Picture)

We want to build a model that predicts:

👉 Will a customer purchase or not?

Based on:

- Age
- Salary
- Experience
- Gender

This is a Binary Classification Problem.

Target:

- 0 = Not Purchased
- 1 = Purchased

### Step 1 — Create Dataset

In [5]:
import pandas as pd

data = {
    "Age": [22, 25, 35, 52, 46, 56, 23, 30, 40, 60],
    "Salary": [20000, 25000, 70000, 75000, 65000, 90000, 22000, 40000, 50000, 95000],
    "Experience": [1, 2, 10, 25, 18, 30, 1, 5, 10, 35],
    "Gender": ["Male", "Female", "Female", "Male", "Male", "Male", "Female", "Male", "Female", "Male"],
    "Purchased": [0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
}

df = pd.DataFrame(data)

In [3]:
data = {
    "Age": [...],
    "Salary": [...],
    "Experience": [...],
    "Gender": [...],
    "Purchased": [...]
}

We convert dictionary → DataFrame.

In [6]:
df = pd.DataFrame(data)

Concept

A DataFrame is a table like Excel.
```
| Age | Salary | Experience | Gender | Purchased |
| --- | ------ | ---------- | ------ | --------- |
```
This is your training data.

### Step 2 — Encoding (Very Important)

Machine Learning models cannot understand text.

So we convert:
```
Male → 0  
Female → 1
```

In [7]:
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

Concept

This is called Label Encoding.

Models work with numbers only.

### Step 3 — Feature Engineering (Real Data Science)

We created a new feature:

In [8]:
df['Salary_per_Experience'] = df['Salary'] / df['Experience']

Why we did this?

Two people:

- Person A: Salary 60k, Exp 10 yrs
- Person B: Salary 60k, Exp 2 yrs

Both salary same… BUT second person earns faster.

This new feature gives hidden intelligence to model.

👉 This is what real data scientists do.

### Step 4 — Train Test Split

```
X = features
y = target
```
Then split:

In [10]:
# Import the function properly from scikit-learn:
from sklearn.model_selection import train_test_split

# Call it with underscores, not spaces:
X = df[['Age', 'Salary', 'Experience', 'Gender', 'Salary_per_Experience']]
y = df['Purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


 Concept

We divide data into:
| Data Type           | Purpose      |
| ------------------- | ------------ |
| Training Data (80%) | Model learns |
| Testing Data (20%)  | Model exam   |

Think like:
- Study material 
- Final exam 

We never test on training data.

### Step 5 — Pipeline + GridSearch

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=5)

grid.fit(X_train, y_train)

C:\Users\DELL XPS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', LogisticRegression())]),
             param_grid={'model__C': [0.01, 0.1, 1, 10]})

### Why Pipeline?

Normally steps are:

1. Scale data
2. Train model
3. Predict

Pipeline makes it automatic.

Whenever we call .fit() or .predict():
```
Data → Scaling → Model → Prediction
```
his prevents mistakes in deployment.

### Why Scaling?
```
StandardScaler()
```
Features have different ranges:
| Feature | Range      |
| ------- | ---------- |
| Age     | 20–60      |
| Salary  | 20k–95k  |

Large numbers dominate model.

Scaling makes everything comparable.

### GridSearchCV (Hyperparameter Tuning)
```
param_grid = {'model__C': [0.01, 0.1, 1, 10]}
```
Big Concept

We don’t know best model settings.

So GridSearch tries ALL options:
```
| C value | Score |
| ------- | ----- |
| 0.01    | ?     |
| 0.1     | ?     |
| 1       | ?     |
| 10      | ?     |

Then selects BEST one automatically.

### What is CV=5?

Cross Validation = 5 mini exams.

Instead of 1 test → we test 5 times for reliability.

This makes model trustworthy.

### Step 6: Best Model
```
best_model = grid.best_estimator_
```
Now we have:

- Best parameters
- Best trained model
```
grid.best_params_
grid.best_score_
```

In [13]:
best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("CV Score:", grid.best_score_)

Best Params: {'model__C': 10}
CV Score: 1.0


### Step 7: Evaluation

We check real performance:
```
Accuracy
Confusion Matrix
Precision
Recall
```
Confusion Matrix Meaning
```
|          | Pred 0 | Pred 1 |
| -------- | ------ | ------ |
| Actual 0 | TN     | FP     |
| Actual 1 | FN     | TP     |
```

Key metrics:

Precision

Out of predicted buyers, how many actually bought?

Recall

Out of real buyers, how many we caught?

These are business metrics.

In [14]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

y_pred = best_model.predict(X_test)

print("Accuracy:", best_model.score(X_test, y_test))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

Accuracy: 1.0
Confusion Matrix:
 [[1 0]
 [0 1]]
Precision: 1.0
Recall: 1.0


### Step 8: Save Final Model
```
joblib.dump(best_model, "final_pipeline_model.pkl")
```
Why save?

Because training takes time.

Later we just load model instantly.

This is how ML apps work.

In [15]:
import joblib

joblib.dump(best_model, "final_pipeline_model.pkl")

print("Final model saved!")

Final model saved!


### Step 9: Prediction System
```
model = joblib.load("final_pipeline_model.pkl")
```
Predict new customer:
new_data = [[35, 60000, 10, 1, 6000]]
prediction = model.predict(new_data)
```
Real-world prediction system ready.

In [16]:
# Load model
model = joblib.load("final_pipeline_model.pkl")

# New input
new_data = [[35, 60000, 10, 1, 6000]]

prediction = model.predict(new_data)

print("Final Prediction:", prediction)

Final Prediction: [1]


C:\Users\DELL XPS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


What You Built

👉 A complete ML system:

- Clean data
- Feature engineering
- Pipeline automation
- Hyperparameter tuning
- Evaluation metrics
-  Deployment-ready model


### Project Structure (GitHub Ready)
```
customer_purchase_project/
│── train.py
│── predict.py
│── final_pipeline_model.pkl
│── README.md
```